# Vérification de l'environnement

Ce notebook répond à une seule question : **ce poste peut-il exécuter la chaîne du projet, et
sur quoi exactement ?** Il ne mesure rien de scientifique et ne produit aucun chiffre destiné au
rapport. Il existe pour qu'une panne se déclare ici, en quelques secondes, plutôt qu'au bout de
quarante minutes d'entraînement, au démarrage d'un conteneur ou dans un job de CI.

Chaque section vérifie une chose dont l'absence a une conséquence nommée. Une vérification dont
l'échec ne changerait rien n'a pas sa place ici.

Trois verdicts sont possibles, et la distinction entre les deux derniers est la seule qui compte
en pratique :

- `OK` : la vérification passe.
- `ALERTE` : un manque, qui bloque une partie du travail seulement. Le corpus absent en est le cas
  type : il empêche d'entraîner, pas de servir l'API. Un poste de développement backend peut vivre
  avec plusieurs alertes.
- `ECHEC` : une incohérence, pas un manque. Une version installée hors des bornes déclarées, une
  empreinte de corpus qui ne correspond plus, un tirage aléatoire non reproductible. Un `ECHEC`
  invalide un résultat, ou en produira un faux.

Le notebook ne fait **aucun appel réseau et n'écrit aucun fichier**. Il tourne donc hors ligne et
sur un environnement incomplet, ce qui est précisément la situation où on en a besoin.

In [ ]:
from __future__ import annotations

import math
import os
import platform
import random
import sys
import time
import tomllib
from collections import Counter
from importlib import metadata
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OK = "OK"
ALERTE = "ALERTE"
ECHEC = "ECHEC"

#: Chaque verification enregistree, dans l'ordre ou elle a ete faite.
CHECKS: list[tuple[str, str, str]] = []


def record(label: str, state: str, detail: str = "") -> str:
    """Record one verification and print its verdict.

    Args:
        label: What was verified.
        state: One of ``OK``, ``ALERTE`` or ``ECHEC``.
        detail: The measured value, printed next to the verdict. It states what
            was observed, never why it matters: the reasoning belongs to the
            prose, so that a verdict line stays readable when it passes.

    Returns:
        The recorded state.
    """
    CHECKS.append((label, state, detail))
    print(f"[{state:^6}] {label:<46} {detail}")
    return state


def check(label: str, passed: bool, detail: str = "", *, on_failure: str = ECHEC) -> str:
    """Record a boolean verification.

    Args:
        label: What was verified.
        passed: Outcome of the verification.
        detail: The measured value.
        on_failure: State recorded when the verification fails. Pass ``ALERTE``
            for what blocks part of the work only.

    Returns:
        The recorded state.
    """
    return record(label, OK if passed else on_failure, detail)


print(f"racine du depot : {REPO_ROOT}")

## 1. Interpréteur et environnement virtuel

La borne de version n'est pas recopiée ici : elle est lue dans `pyproject.toml`. Une contrainte
écrite deux fois est une contrainte qui finit par diverger, et c'est le notebook qui aurait tort.

L'environnement virtuel mérite sa propre vérification parce que son absence se manifeste ailleurs,
sous un autre nom. Le hook `mypy` tourne en `language: system`, dans l'environnement du projet et
non dans un environnement isolé, parce que dupliquer PyTorch dans un environnement de hook serait
absurde. Un terminal sans `.venv` actif fait donc échouer le commit sur un message de typage, qui
ne dit rien du vrai problème.

In [ ]:
from packaging.specifiers import SpecifierSet

PYPROJECT = tomllib.loads((REPO_ROOT / "pyproject.toml").read_text(encoding="utf-8"))
REQUIRES_PYTHON = SpecifierSet(PYPROJECT["project"]["requires-python"])

running = platform.python_version()
check(
    "version de Python dans les bornes declarees",
    REQUIRES_PYTHON.contains(running),
    f"{running}, declare : {REQUIRES_PYTHON}",
)

in_venv = sys.prefix != sys.base_prefix
check("environnement virtuel actif", in_venv, sys.prefix)
check(
    "environnement virtuel du depot",
    in_venv and Path(sys.prefix).resolve() == (REPO_ROOT / ".venv").resolve(),
    f"attendu : {REPO_ROOT / '.venv'}",
    on_failure=ALERTE,
)

print()
print(f"interpreteur      {sys.executable}")
print(f"plateforme        {platform.platform()}")
print(f"machine           {platform.machine()}")

## 2. Dépendances déclarées contre dépendances installées

Les groupes sont lus dans `pyproject.toml` et comparés à ce que `importlib.metadata` voit
réellement installé. Là encore, rien n'est recopié.

La règle de verdict traduit une intention du projet, elle n'est pas arbitraire :

- Un paquet **absent** vaut `ALERTE`. Les groupes sont séparés pour être installés séparément :
  l'image backend n'installe que `api`, l'image d'entraînement que `train`, et le notebook d'EDA
  demande `eda`. Un groupe non installé est un choix légitime, pas un défaut.
- Un paquet **installé hors des bornes** vaut `ECHEC`. Ce n'est plus un choix d'installation,
  c'est une dérive : le code tourne contre une version que le dépôt ne déclare pas, et le plancher
  `torch>=2.13` n'est pas une préférence de confort mais la réponse à GHSA-rrmf-rvhw-rf47.

In [ ]:
from packaging.requirements import Requirement

GROUPS: dict[str, list[str]] = {
    "base": PYPROJECT["project"]["dependencies"],
    **PYPROJECT["project"]["optional-dependencies"],
}


def audit_group(requirements: list[str]) -> tuple[list[str], list[str]]:
    """Compare declared requirements with what is installed.

    Args:
        requirements: Requirement strings, as declared in ``pyproject.toml``.

    Returns:
        A pair ``(missing, conflicting)``. A missing distribution is not
        installed at all; a conflicting one is installed at a version the
        declared specifier refuses.
    """
    missing: list[str] = []
    conflicting: list[str] = []
    for raw in requirements:
        requirement = Requirement(raw)
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            missing.append(requirement.name)
            continue
        if not requirement.specifier.contains(installed, prereleases=True):
            conflicting.append(f"{requirement.name} {installed} hors de {requirement.specifier}")
    return missing, conflicting


for group, requirements in GROUPS.items():
    missing, conflicting = audit_group(requirements)
    state = ECHEC if conflicting else ALERTE if missing else OK
    record(
        f"groupe {group}",
        state,
        f"{len(requirements) - len(missing)}/{len(requirements)} installes",
    )
    for line in conflicting:
        print(f"         conflit : {line}")
    if missing:
        print(f"         absents : {', '.join(missing)}")

## 3. Torch, matériel et exécution réelle

`describe_hardware()` est la fonction que chaque run enregistre dans MLflow. La lire ici plutôt
que d'interroger `torch` directement garantit que ce notebook décrit le poste dans les mêmes
termes qu'un run tracé, et qu'une durée reportée reste interprétable.

Deux choses sont vérifiées séparément, parce qu'elles échouent séparément :

Le **périphérique retenu** par `resolve_device("auto")`. Sur CPU, tout fonctionne mais les durées
n'ont plus rien de comparable à celles de `reports/`, et la précision mixte est inutile : sur CPU
elle ajoute du coût de conversion sans gain. D'où l'alerte, pas l'échec.

Le **fait que torch calcule**. Un import réussi ne prouve rien : sous Windows, une DLL CUDA
manquante ou un runtime Visual C++ absent laisse `import torch` passer et fait échouer la première
opération. Un produit matriciel de 512 sur 512 tranche la question en quelques millisecondes.

In [ ]:
import torch

from src.utils.device import describe_hardware, resolve_device

for key, value in describe_hardware().items():
    print(f"{key:<18} {value}")
print()

device = resolve_device("auto")
check("acceleration GPU", device.type == "cuda", f"peripherique retenu : {device}", on_failure=ALERTE)

started = time.perf_counter()
matrix = torch.randn(512, 512, device=device)
total = float((matrix @ matrix.T).sum())
elapsed = time.perf_counter() - started
check(
    "torch execute un produit matriciel",
    math.isfinite(total),
    f"512x512 en {elapsed * 1000:.0f} ms sur {device}",
)

## 4. Graine et déterminisme

`set_seed` est appelée une fois par expérience, avant la construction des chargeurs et du modèle.
Toute la reproductibilité du projet en dépend : sans elle, l'ablation sur la taille du corpus
n'est plus rejouable et la comparaison entre les deux familles de modèles perd son sens.

La vérification ici n'est pas déclarative, c'est un test : on ensemence, on tire une valeur de
chacun des trois générateurs, on ré-ensemence, on retire. Les deux tirages doivent être identiques.
C'est la seule façon de constater sur ce poste précis que la chaîne d'ensemencement fonctionne,
plutôt que de faire confiance au fait qu'elle est écrite quelque part.

In [ ]:
import numpy as np

from src.utils.seed import DEFAULT_SEED, set_seed


def draw() -> tuple[float, float, float]:
    """Draw one value from each generator seeded by ``set_seed``.

    Returns:
        One draw from ``random``, from ``numpy`` and from ``torch``.
    """
    return random.random(), float(np.random.rand()), float(torch.randn(1))


state = set_seed(DEFAULT_SEED)
first = draw()
set_seed(DEFAULT_SEED)
second = draw()

check("un meme ensemencement redonne le meme tirage", first == second, f"graine {state.seed}")
check(
    "PYTHONHASHSEED fixe",
    os.environ.get("PYTHONHASHSEED") == str(state.seed),
    os.environ.get("PYTHONHASHSEED", "absent"),
)
check(
    "cuDNN en mode deterministe",
    bool(torch.backends.cudnn.deterministic),
    f"benchmark : {torch.backends.cudnn.benchmark}",
)

## 5. Corpus figé

Le corpus n'est pas seulement présent ou absent : il est **le bon ou un autre**. Un fichier
`train.jsonl` reconstruit par une exécution ultérieure de la préparation, ou tronqué par une copie
interrompue, ressemble en tout point au corpus d'origine sur un listing de répertoire.

Les empreintes du manifeste tranchent. Elles sont recalculées ici avec la fonction `checksum` qui
les a écrites, sur les exemples relus par le même chargeur que l'entraînement, et comparées à ce
que le manifeste déclare. C'est la vérification la plus lente du notebook, quelques secondes pour
une cinquantaine de mégaoctets, et c'est la seule qui garantisse que les scores de `reports/` et
un entraînement lancé aujourd'hui portent sur le même corpus.

Un corpus absent n'est pas un échec : le poste peut servir l'API sans lui. La commande qui le
construit est rappelée dans le message.

In [ ]:
from src.data.dataset import MANIFEST_NAME, load_manifest, load_split
from src.data.split import checksum

PROCESSED = REPO_ROOT / "data" / "processed" / "xsum"

if not (PROCESSED / MANIFEST_NAME).is_file():
    check(
        "corpus de travail",
        False,
        f"absent de {PROCESSED}, le construire avec : python -m src.data.build",
        on_failure=ALERTE,
    )
else:
    manifest = load_manifest(PROCESSED)
    print(f"corpus            {manifest.name} (source : {manifest.source_dataset})")
    print(f"version           {manifest.dataset_version[:16]}...")
    print(f"graine du tirage  {manifest.seed}")
    print(f"tokenizer         {manifest.tokenizer}")
    print()

    started = time.perf_counter()
    for split, expected in manifest.splits.items():
        examples = load_split(PROCESSED, split)
        intact = checksum(examples) == manifest.split_checksums[split]
        check(
            f"split {split}",
            len(examples) == expected and intact,
            f"{len(examples)}/{expected} exemples, empreinte {'conforme' if intact else 'DIFFERENTE'}",
        )
    print()
    print(f"empreintes recalculees en {time.perf_counter() - started:.1f} s")

## 6. Configurations d'expériences

`discover_experiments` refuse un fichier invalide et refuse deux fichiers qui déclarent le même
nom, parce que deux runs homonymes écriraient dans le même répertoire et que le second effacerait
le premier en silence. Le chargement est aussi ce qui fait passer les validateurs de cohérence :
un run zéro-shot qui déclarerait une proportion de corpus, ou un run entraîné qui n'en déclarerait
pas, est rejeté ici.

Ces erreurs sont bon marché à ce moment, et coûteuses au milieu d'une campagne.

In [ ]:
from src.experiments.config import discover_experiments

experiments = []
try:
    experiments = discover_experiments(REPO_ROOT / "configs" / "experiments")
except (OSError, ValueError) as error:
    check("configurations d'experiences valides", False, str(error).splitlines()[0])
else:
    check("configurations d'experiences valides", True, f"{len(experiments)} fichiers charges")

print()
for config in experiments:
    proportion = (
        "sans entrainement"
        if config.dataset.percentage is None
        else f"{config.dataset.percentage:>3} % du corpus"
    )
    print(f"  {config.experiment.name:<24} {config.model.type:<12} {proportion}")

## 7. Résultats enregistrés

Une expérience déclarée sans enregistrement porte le statut `NOT_RUN`, jamais une valeur estimée.
Cette section applique la même règle au poste : elle dit lesquelles des expériences déclarées ont
un `run.json` sous `reports/results/`, et à quel statut.

C'est ce qui distingue un dépôt fraîchement cloné, où les tableaux du rapport sont lisibles mais
non reproduits localement, d'un poste où la campagne a effectivement tourné. Un run plafonné porte
`PARTIAL` et n'entre dans aucun tableau : `find_record` privilégie l'enregistrement complet, de
sorte qu'une répétition rapide ne masque jamais la mesure qu'elle préparait.

In [ ]:
from src.experiments.record import STATUS_NOT_RUN, STATUS_OK, find_record

RESULTS = REPO_ROOT / "reports" / "results"

complete = 0
for config in experiments:
    found = find_record(RESULTS, config.experiment.name)
    status = STATUS_NOT_RUN if found is None else found.status
    complete += int(status == STATUS_OK)
    print(f"  {config.experiment.name:<24} {status}")

print()
check(
    "chaque experience declaree a un enregistrement complet",
    bool(experiments) and complete == len(experiments),
    f"{complete}/{len(experiments)} au statut {STATUS_OK}",
    on_failure=ALERTE,
)

## 8. Réglages du backend

Les réglages sont chargés depuis le `.env` de la racine du dépôt, explicitement, et non par
`get_settings()`. La raison est que `Settings` lit `.env` relativement au répertoire courant :
appelée depuis `notebooks/`, elle ne verrait pas le fichier que l'API lit au démarrage, et le
notebook rapporterait des valeurs par défaut là où l'application en utilise d'autres. Vérifier
l'environnement avec une lecture qui n'est pas celle de l'application n'aurait aucun intérêt.

Un `.env` absent vaut alerte : les valeurs par défaut de `Settings` s'appliquent et elles sont
utilisables en local.

La vérification qui compte dans cette section est la dernière. `SYNTRA_DEVICE=cuda` sur un poste
sans GPU visible est accepté par la validation du modèle, puis lève au démarrage de l'API, quand
`resolve_device` refuse un CUDA explicite qu'il ne peut pas honorer. La parité entre `Settings` et
`.env.example`, elle, n'est pas revérifiée ici : `scripts/check_sync.py` la contrôle déjà.

In [ ]:
from backend.app.core.config import Settings

settings = Settings(_env_file=REPO_ROOT / ".env")

check("fichier .env", (REPO_ROOT / ".env").is_file(), str(REPO_ROOT / ".env"), on_failure=ALERTE)

print()
print(f"environnement       {settings.environment}")
print(f"peripherique        {settings.device}")
print(f"registre            {settings.registry_backend} sous {settings.registry_root}")
print(f"modele pre-entraine {settings.pretrained_model_id}")
print(f"MLflow              {settings.mlflow_tracking_uri} / {settings.mlflow_experiment}")
print()

try:
    resolved = resolve_device(settings.device)
except (RuntimeError, ValueError) as error:
    check("SYNTRA_DEVICE utilisable sur ce poste", False, str(error))
else:
    check("SYNTRA_DEVICE utilisable sur ce poste", True, f"{settings.device} donne {resolved}")

## 9. Magasins locaux

Présence des répertoires que la chaîne lit et écrit. Tout est vérifié sur le système de fichiers,
sans joindre le serveur MLflow : un contrôle d'environnement qui dépend du réseau devient un
contrôle du réseau, et il faudrait alors distinguer un serveur éteint d'un poste mal installé.
Interroger `mlflow_tracking_uri` est le travail de `make mlflow-ui`, pas celui de ce notebook.

Chacun de ces manques vaut alerte : ils suivent l'usage réel du poste. Un poste qui n'a jamais
entraîné n'a ni `runs/` ni `mlruns/`, et cela ne l'empêche pas de servir l'API ni de lancer les
tests.

In [ ]:
STORES = {
    "magasin MLflow local": REPO_ROOT / "mlflow.db",
    "runs MLflow": REPO_ROOT / "mlruns",
    "registre de modeles": REPO_ROOT / settings.registry_root,
    "resultats d'experiences": REPO_ROOT / "reports" / "results",
    "figures du rapport": REPO_ROOT / "reports" / "figures",
    "checkpoints d'entrainement": REPO_ROOT / "runs",
}

for label, path in STORES.items():
    check(label, path.exists(), str(path), on_failure=ALERTE)

## 10. Provenance du code

`describe_provenance()` lit le commit dans git plutôt que dans une constante entretenue à la main,
et c'est ce qui empêche les deux de diverger. Chaque run trace ce commit à côté de ses métriques.

Un arbre de travail modifié n'est pas refusé, il est signalé. Refuser de tracer un run parti d'une
modification non commitée sortirait la mesure du magasin de suivi, ce qui est exactement l'inverse
de ce que la traçabilité sert à faire. L'alerte dit seulement qu'un score produit maintenant ne
sera pas rattachable à un état du dépôt qu'un tiers puisse retrouver.

In [ ]:
from src.tracking.provenance import UNKNOWN, describe_provenance

provenance = describe_provenance(REPO_ROOT)
for key, value in provenance.to_dict().items():
    print(f"{key:<12} {value}")
print()

check("commit connu", provenance.commit != UNKNOWN, provenance.commit[:12])
check(
    "arbre de travail propre",
    provenance.dirty is False,
    UNKNOWN if provenance.dirty is None else str(provenance.dirty).lower(),
    on_failure=ALERTE,
)

## 11. Verdict

Le compte, puis la liste de ce qui n'est pas `OK`. Rien d'autre : les lignes qui passent ont déjà
été affichées au fil des sections, les répéter ne dirait rien de plus.

In [ ]:
counts = Counter(state for _, state, _ in CHECKS)
print(
    f"{len(CHECKS)} verifications : "
    f"{counts[OK]} {OK}, {counts[ALERTE]} {ALERTE}, {counts[ECHEC]} {ECHEC}"
)
print()

for label, state, detail in CHECKS:
    if state != OK:
        print(f"[{state:^6}] {label:<46} {detail}")

if not counts[ALERTE] and not counts[ECHEC]:
    print("Rien a signaler : la chaine complete peut etre lancee depuis ce poste.")

## Ce que ce notebook ne vérifie pas

Le dire explicitement évite de lui prêter une garantie qu'il n'apporte pas.

Il ne joint aucun service : ni le serveur MLflow, ni Hugging Face, ni un registre d'images, ni
l'API elle-même. Un `t5-small` absent du cache local ne sera constaté qu'au premier
téléchargement, et la révision épinglée `df1b051c` n'est vérifiable que par ce téléchargement.

Il n'exécute ni entraînement ni évaluation. Une mémoire GPU suffisante pour un produit matriciel
de 512 sur 512 ne dit rien de ce que demande un `batch_size` donné, et le seul contrôle honnête de
ce point est `make reproduce MODE=quick`.

Il ne remplace pas les hooks. `make lint`, `make typecheck` et `make test` portent sur le code, ce
notebook porte sur ce qui l'entoure. Les deux échouent pour des raisons différentes, et une chaîne
verte ici avec un `mypy` rouge reste une chaîne rouge.